# Trading agent: cloud training on Google Colab

Research and paper trading only. This notebook never places orders and needs no API keys.

**One-time setup**
1. On your PC, in the `trading-agent` folder, run:
   `git archive --format=zip -o trading-agent.zip HEAD`
2. Upload `trading-agent.zip` to the top level of your Google Drive (**My Drive**).
3. Open this notebook in Colab (**File → Upload notebook**), then **Runtime → Run all**.

Downloaded data and results are kept in **My Drive / trading-agent-data**, so they
survive Colab restarts. You can also copy them to your PC from there.

## 1. Set up the project

In [ ]:
import os, sys, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = pathlib.Path("/content/drive/MyDrive")
    PROJECT = pathlib.Path("/content/trading-agent")
    !unzip -o -q "{DRIVE}/trading-agent.zip" -d "{PROJECT}"
    # Keep downloaded data and results on Drive so they survive restarts.
    (DRIVE / "trading-agent-data").mkdir(exist_ok=True)
    if not (PROJECT / "data").exists():
        os.symlink(DRIVE / "trading-agent-data", PROJECT / "data")
    os.chdir(PROJECT)
    !pip install -q -r requirements.txt
elif pathlib.Path.cwd().name == "notebooks":
    os.chdir("..")  # running on your PC: the project is one folder up

sys.path.insert(0, os.getcwd())
print("Project folder:", os.getcwd())

## 2. Run the automated tests
Everything should pass before you trust any result.

In [ ]:
!{sys.executable} -m pytest -q

## 3. Download data

**Crypto:** Colab runs on US servers, where binance.com is blocked, so we use Binance.US.
If that fails, try `"coinbase"` or `"kraken"`.

**Forex:** free 1-minute files from HistData.com. Years already downloaded are skipped.

In [ ]:
CRYPTO_EXCHANGE = "binanceus"
!{sys.executable} -m trading_agent.download crypto --exchange {CRYPTO_EXCHANGE}

In [ ]:
!{sys.executable} -m trading_agent.download histdata --years 2019-2025

## 4. Choose a market to study

In [ ]:
from trading_agent.config import load_config
from trading_agent.market_data.storage import data_path, load_ohlcv

MARKET, SYMBOL = "forex", "EURUSD"  # e.g. "crypto", "BTC/USDT"

cfg = load_config()
df = load_ohlcv(data_path(MARKET, SYMBOL, cfg[MARKET]["timeframe"]))
print(f"{len(df):,} candles from {df.index[0]} to {df.index[-1]}")
df["close"].plot(title=SYMBOL, figsize=(10, 3));

## 5. Compare models with walk-forward testing

Each model is trained on the past and tested on the next slice of time, five times.
Look at **folds_profitable** and **avg_return** after costs, not just accuracy.

In [ ]:
from trading_agent.backtest.engine import Costs
from trading_agent.evaluation import compare_models
from trading_agent.experiment import risk_from_config

costs = Costs(**cfg["costs"][MARKET])
risk = risk_from_config(cfg["risk"])
folds, summary = compare_models(df, ["logistic", "gboost"], cfg["model"], costs, risk, n_folds=5)
display(folds)
display(summary)

## 6. Equity curve on the unseen test period

In [ ]:
import pandas as pd
from trading_agent.backtest.engine import RiskRules, run_backtest
from trading_agent.features import build_dataset
from trading_agent.models.baseline import make_model, probabilities_to_signal, time_split
from trading_agent.strategies import buy_and_hold

MODEL_KIND = "logistic"
horizon = cfg["model"]["horizon"]
X, y = build_dataset(df, horizon)
X_train, X_test, y_train, y_test = time_split(X, y, cfg["model"]["test_fraction"], gap=horizon + 1)
model = make_model(MODEL_KIND).fit(X_train, y_train)
proba = pd.Series(model.predict_proba(X_test)[:, 1], index=X_test.index)
signal = probabilities_to_signal(proba, cfg["model"]["threshold"], cfg["model"]["allow_short"])

test_df = df.loc[X_test.index[0]:]
ax = run_backtest(test_df, signal, costs, risk).equity.plot(label=f"{MODEL_KIND} model", figsize=(10, 4))
run_backtest(test_df, buy_and_hold(test_df), costs, RiskRules()).equity.plot(ax=ax, label="buy & hold")
ax.set_title(f"{SYMBOL}: unseen test period, after costs")
ax.set_ylabel("account value")
ax.legend();

## 7. Save results

Saved under `data/results/` (in Colab that is **My Drive / trading-agent-data / results**).
Only load `.joblib` model files you created yourself: they can run code when opened.

In [ ]:
import joblib

results_dir = pathlib.Path("data/results")
results_dir.mkdir(parents=True, exist_ok=True)
safe_symbol = SYMBOL.replace("/", "-")
folds.to_csv(results_dir / f"walk_forward_{MARKET}_{safe_symbol}.csv", index=False)
joblib.dump(model, results_dir / f"{MODEL_KIND}_{MARKET}_{safe_symbol}.joblib")
print("Saved to", results_dir.resolve())

## Next steps
- Change `MARKET`/`SYMBOL` in step 4 and re-run steps 4 to 7 for other markets.
- To paper trade crypto on your PC, copy `My Drive/trading-agent-data/raw/crypto/*.csv`
  into your PC's `trading-agent/data/raw/crypto/`, then run
  `python -m trading_agent.paper_trade --market crypto --start 2025-01-01`.